In [1]:
import torch
from torch import nn
import numpy as np
from torchvision.datasets import ImageFolder
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

In [2]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [3]:
weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

In [4]:
train_ds = ImageFolder(root="faces/train", transform=preprocess)
valid_ds = ImageFolder(root="faces/dev", transform=preprocess)

In [5]:
mini_batch_size = 64
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=mini_batch_size, shuffle=True, drop_last=False, num_workers=4)
valid_dl = torch.utils.data.DataLoader(valid_ds, batch_size=mini_batch_size, num_workers=4)

In [6]:
model = mobilenet_v3_small(weights=weights)

In [7]:
model

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

In [8]:
model.classifier = nn.Sequential(nn.Linear(576, 1024), nn.Hardswish(), nn.Dropout(0.2, inplace=True), nn.Linear(1024, 7001))

In [9]:
model = model.to(device)
model

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

In [10]:
class WrappedDataLoader:
    def __init__(self, dl, func):
        self.dl = dl
        self.func = func

    def __len__(self):
        return len(self.dl)

    def __iter__(self):
        for b in self.dl:
            yield (self.func(*b))


def put_to_gpu(x, y):
    return x.to(device), y.to(device)

In [11]:
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

In [12]:
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        for param in m.parameters():
            param.requires_grad = True

In [13]:
for name, param in model.named_parameters():
    if "9" in name or "10" in name or "11" in name or "12" in name:
        param.requires_grad = True

In [14]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

8507985

In [15]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

In [16]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

    def load_best_model(self, model):
        model.load_state_dict(self.best_model_state)

In [17]:
early_stopping = EarlyStopping(patience=3, delta=0.01)

In [18]:
def fit(epochs, model, optimizer, train_dl, valid_dl=None):
    loss_func = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        for X_mb, y_mb in train_dl:
            optimizer.zero_grad()

            y_hat = model(X_mb)

            loss = loss_func(y_hat, y_mb)
            loss.backward()

            optimizer.step()

        model.eval()

        train_loss = 0.0
        with torch.no_grad():
            for X_mb, y_mb in train_dl:
                out = model(X_mb)
                loss = loss_func(out, y_mb)
                train_loss += loss.item()
        train_loss /= len(train_dl)
        print('epoch {}, training loss {}'.format(epoch + 1, train_loss))
        valid_loss = 0.0
        with torch.no_grad():
            for X_mb, y_mb in valid_dl:
                out = model(X_mb)
                loss = loss_func(out, y_mb)
                valid_loss += loss.item()
        valid_loss /= len(valid_dl)
        print('epoch {}, validation loss {}'.format(epoch + 1, valid_loss))

        early_stopping(valid_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

    print('Finished training')

    return model

In [19]:
epochs = 10

model = fit(epochs, model, optimizer, WrappedDataLoader(train_dl, put_to_gpu), WrappedDataLoader(valid_dl, put_to_gpu))

epoch 1, training loss 4.610305543573508
epoch 1, validation loss 5.2160336208517855
epoch 2, training loss 2.5952487372175215
epoch 2, validation loss 3.715581981965785
epoch 3, training loss 1.841024241674099
epoch 3, validation loss 3.3532913661744086
epoch 4, training loss 1.2895250706448198
epoch 4, validation loss 3.374946565035274
epoch 5, training loss 0.9893844558481106
epoch 5, validation loss 3.499092072847774
epoch 6, training loss 0.6055670617253841
epoch 6, validation loss 3.6155187102950688
Early stopping
Finished training


In [20]:
early_stopping.load_best_model(model)
model = model.cpu()

In [21]:
def evaluate(model, data_loader):    
    model.eval()
    accuracy = 0
    with torch.no_grad():
        for X, y in data_loader:
            y_hat = model(X).cpu().numpy()
            y_hat = np.argmax(y_hat, axis=1)
            accuracy += (y_hat == y.cpu().numpy()).mean()
    accuracy /= len(data_loader)

    return accuracy

In [22]:
evaluate(model.to(device), WrappedDataLoader(train_dl, put_to_gpu))

np.float64(0.8629538523062861)

In [23]:
evaluate(model.to(device), WrappedDataLoader(valid_dl, put_to_gpu))

np.float64(0.41385622396379657)

In [24]:
torch.save(model.state_dict(), "classification_finetuned.pth")